# 01 · Wstępne przetwarzanie danych eye-trackingowych

**Kurs:** Praktyczna Analiza Danych Eye-Tracking i EEG w Pythonie

W tym notebooku przejdziemy całą drogę od **surowych próbek** zarejestrowanych przez eye-tracker do **listy fiksacji i sakad**:

1. wczytanie i kontrola jakości surowych danych (czas, X, Y, średnica źrenicy),
2. wykrycie mrugnięć i interpolacja brakujących danych,
3. wygładzanie sygnału,
4. obliczenie prędkości kątowej ruchu oka,
5. detekcja fiksacji i sakad algorytmem **I-VT** (*Velocity-Threshold Identification*),
6. walidacja wyników (główna sekwencja sakad, wpływ progu, porównanie z „prawdą”).

**Dane:** 10-sekundowe nagranie (500 Hz) osoby oglądającej plakat reklamowy. Dane są symulowane, ale mają wszystkie typowe „problemy” prawdziwych nagrań: szum pomiarowy, mrugnięcia z artefaktami na brzegach i krótkie utraty sygnału. Dzięki symulacji znamy też **prawdziwe** położenie fiksacji i możemy sprawdzić, jak dobrze działa nasz algorytm.

**Szacowany czas:** ok. 90 minut.

> **Jak pracować z notebookiem?** Komórki uruchamiamy po kolei klawiszami **Shift+Enter**. Przy pierwszym uruchomieniu VS Code poprosi o wybór jądra (*Select Kernel*) – wybierz *Python Environments → Python 3.10*. Jeśli coś pójdzie nie tak, zrestartuj jądro (*Restart*) i uruchom wszystkie komórki od początku (*Run All*). Miejsca oznaczone ✏️ zachęcają do samodzielnych modyfikacji.

## 0. Przygotowanie środowiska

Importujemy biblioteki i ustalamy ścieżki do katalogów. Notebooki leżą w katalogu `notebooks/`, dane w `dane/`, a wyniki zapisujemy w `wyniki/` (ten katalog nie trafia do repozytorium).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from scipy.signal import butter, filtfilt, savgol_filter

# Ścieżki: działają niezależnie od tego, czy notebook uruchomiono z katalogu notebooks/, czy z katalogu głównego
KATALOG_GLOWNY = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DANE = KATALOG_GLOWNY / "dane"
WYNIKI = KATALOG_GLOWNY / "wyniki"
WYNIKI.mkdir(exist_ok=True)

# Wspólny wygląd wykresów; kolory czytelne także dla osób z zaburzeniami widzenia barw
NIEBIESKI, POMARANCZOWY, MORSKI = "#2a78d6", "#eb6834", "#1baf7a"
plt.rcParams.update({"figure.figsize": (12, 4), "axes.grid": True, "grid.alpha": 0.3,
                     "axes.prop_cycle": plt.cycler(color=[NIEBIESKI, POMARANCZOWY, MORSKI])})
print("Dane wczytujemy z katalogu:", DANE)

## 1. Parametry stanowiska i kąt widzenia

Eye-tracker zapisuje położenie wzroku w **pikselach ekranu**. Do analizy ruchów oczu potrzebujemy jednak jednostek niezależnych od monitora – **stopni kąta widzenia** (°). Obiekt o rozmiarze $s$ oglądany z odległości $d$ zajmuje kąt

$$\theta = 2 \cdot \arctan\left(\frac{s}{2d}\right).$$

Dla orientacji: paznokieć kciuka na wyciągniętej ręce to ok. 1,5–2°, a dołek środkowy siatkówki (obszar najostrzejszego widzenia) obejmuje zaledwie ok. 1–2°. Parametry stanowiska zawsze zapisujemy w protokole badania.

In [ ]:
# Parametry stanowiska badawczego (z protokołu badania)
EKRAN_SZER_PX, EKRAN_WYS_PX = 1920, 1080  # rozdzielczość monitora [px]
EKRAN_SZER_CM, EKRAN_WYS_CM = 53.1, 29.9  # fizyczny rozmiar ekranu 24" [cm]
ODLEGLOSC_CM = 60.0  # odległość oczu od ekranu [cm]
FS = 500  # częstotliwość próbkowania eye-trackera [Hz]


def px_na_stopnie(x_px, y_px):
    """Przelicza współrzędne ekranu [px] na kąt widzenia [°] względem środka ekranu."""
    x_cm = (np.asarray(x_px) - EKRAN_SZER_PX / 2) * EKRAN_SZER_CM / EKRAN_SZER_PX
    y_cm = (np.asarray(y_px) - EKRAN_WYS_PX / 2) * EKRAN_WYS_CM / EKRAN_WYS_PX
    return np.degrees(np.arctan2(x_cm, ODLEGLOSC_CM)), np.degrees(np.arctan2(y_cm, ODLEGLOSC_CM))


PX_NA_STOPIEN = EKRAN_SZER_PX / EKRAN_SZER_CM * ODLEGLOSC_CM * np.tan(np.radians(1))
print(f"W centrum ekranu 1° kąta widzenia ≈ {PX_NA_STOPIEN:.1f} px")
print(f"Cały ekran zajmuje {2 * np.degrees(np.arctan(EKRAN_SZER_CM / 2 / ODLEGLOSC_CM)):.1f}° w poziomie")

## 2. Wczytanie surowych danych

Plik `et_surowe_uczestnik01.csv` zawiera cztery kolumny:

| kolumna | opis |
|---|---|
| `czas_ms` | znacznik czasu eye-trackera [ms] (liczony od włączenia urządzenia) |
| `x_px`, `y_px` | położenie wzroku na ekranie [px]; (0, 0) to **lewy górny** róg |
| `zrenica_mm` | średnica źrenicy [mm] |

Puste pola (`NaN`) oznaczają, że eye-tracker nie znalazł źrenicy – najczęściej z powodu mrugnięcia.

In [ ]:
surowe = pd.read_csv(DANE / "et_surowe_uczestnik01.csv")
surowe.head()

In [ ]:
surowe.info()

In [ ]:
surowe.describe().round(2)

Znaczniki czasu eye-trackera nie zaczynają się od zera, więc tworzymy wygodniejszą oś czasu w sekundach od początku nagrania. Sprawdzamy też, czy próbki są równomiernie rozłożone w czasie – to podstawowy test jakości nagrania.

In [ ]:
surowe["t_s"] = (surowe["czas_ms"] - surowe["czas_ms"].iloc[0]) / 1000

odstepy_ms = np.diff(surowe["czas_ms"])
print("Odstępy między kolejnymi próbkami [ms]:", np.unique(odstepy_ms))
print(f"Częstotliwość próbkowania: {1000 / np.median(odstepy_ms):.0f} Hz")
print(f"Długość nagrania: {surowe['t_s'].iloc[-1]:.2f} s ({len(surowe)} próbek)")

braki = surowe[["x_px", "y_px", "zrenica_mm"]].isna()
print("\nLiczba brakujących próbek w kolumnach:", braki.sum().to_dict())
print(f"Odsetek utraconych danych: {100 * braki['x_px'].mean():.1f}%")

### Pierwszy rzut oka na dane

Wykres położenia wzroku w czasie ma charakterystyczny „schodkowy” kształt: płaskie odcinki to **fiksacje** (oko względnie nieruchome), a strome skoki to **sakady** (szybkie przeniesienia wzroku). Na czerwono zaznaczono brakujące dane.

In [ ]:
fig, osie = plt.subplots(3, 1, figsize=(12, 7), sharex=True)
for os_, kolumna, opis in zip(osie, ["x_px", "y_px", "zrenica_mm"], ["X [px]", "Y [px]", "Źrenica [mm]"]):
    os_.plot(surowe["t_s"], surowe[kolumna], lw=0.8)
    os_.fill_between(surowe["t_s"], 0, 1, where=braki["x_px"], color="red", alpha=0.3,
                     transform=os_.get_xaxis_transform(), label="brak danych")
    os_.set_ylabel(opis)
osie[0].legend(loc="upper right")
osie[-1].set_xlabel("Czas [s]")
fig.suptitle("Surowe dane eye-trackera")
plt.tight_layout()
plt.show()

Ten sam wykres w wersji **interaktywnej** (Plotly) – przybliż myszką okolice mrugnięć i przyjrzyj się, co dzieje się z sygnałem tuż przed i tuż po luce w danych.

In [ ]:
fig = make_subplots(rows=3, cols=1, shared_xaxes=True, vertical_spacing=0.04,
                    subplot_titles=("Położenie X [px]", "Położenie Y [px]", "Średnica źrenicy [mm]"))
for wiersz, kolumna in enumerate(["x_px", "y_px", "zrenica_mm"], start=1):
    fig.add_trace(go.Scattergl(x=surowe["t_s"], y=surowe[kolumna], mode="lines", name=kolumna, line=dict(width=1)),
                  row=wiersz, col=1)
fig.update_xaxes(title_text="Czas [s]", row=3, col=1)
fig.update_layout(height=650, showlegend=False, title="Surowe dane – wersja interaktywna")
fig.show()

Zobaczmy jeszcze, gdzie na ekranie patrzyła osoba badana. Wyraźnie widać skupiska próbek (fiksacje) połączone liniami (sakady).

In [ ]:
obraz = plt.imread(DANE / "bodziec_plakat.png")

fig, ax = plt.subplots(figsize=(12, 6.75))
ax.imshow(obraz, extent=[0, EKRAN_SZER_PX, EKRAN_WYS_PX, 0])
ax.plot(surowe["x_px"], surowe["y_px"], lw=0.6, color=NIEBIESKI, alpha=0.8)
ax.set(xlim=(0, EKRAN_SZER_PX), ylim=(EKRAN_WYS_PX, 0), title="Surowe próbki wzroku na tle bodźca")
ax.grid(False)
plt.show()

## 3. Mrugnięcia i brakujące dane

Podczas mrugnięcia powieka zasłania źrenicę, więc eye-tracker przez ok. 100–300 ms nie rejestruje danych. Problem nie ogranicza się jednak do samej luki: tuż **przed** i tuż **po** mrugnięciu powieka częściowo zasłania źrenicę, przez co:

* mierzona średnica źrenicy gwałtownie (i fałszywie) maleje,
* środek źrenicy jest źle wyznaczany – pozycja wzroku „skacze” (najczęściej w pionie).

Dlatego standardowa procedura obejmuje trzy kroki:

1. **wykrycie luk** w danych i odróżnienie mrugnięć (≥ 75 ms) od krótkich utrat sygnału,
2. **poszerzenie** okna mrugnięcia o margines przed i po luce,
3. **interpolację** usuniętych fragmentów.

Zaczynamy od funkcji pomocniczej, która znajduje ciągłe odcinki spełniające dany warunek. Przyda się nam jeszcze kilka razy.

In [ ]:
def znajdz_odcinki(maska):
    """Zwraca tablicę (n, 2) z indeksami [początek, koniec) kolejnych ciągów wartości True."""
    maska = np.asarray(maska, dtype=int)
    zmiany = np.diff(np.concatenate([[0], maska, [0]]))
    return np.column_stack([np.flatnonzero(zmiany == 1), np.flatnonzero(zmiany == -1)])


# Prosty test na krótkim przykładzie
print(znajdz_odcinki([False, True, True, False, False, True]))

In [ ]:
MIN_CZAS_MRUGNIECIA_MS = 75  # krótsze luki traktujemy jako chwilową utratę sygnału

luki = znajdz_odcinki(surowe["x_px"].isna())
tabela_luk = pd.DataFrame({
    "start_s": surowe["t_s"].to_numpy()[luki[:, 0]],
    "czas_trwania_ms": (luki[:, 1] - luki[:, 0]) * 1000 / FS,
})
tabela_luk["rodzaj"] = np.where(tabela_luk["czas_trwania_ms"] >= MIN_CZAS_MRUGNIECIA_MS,
                                "mrugnięcie", "utrata sygnału")
tabela_luk

Przyjrzyjmy się z bliska pierwszemu mrugnięciu. Zwróć uwagę na brzegi luki: spadek średnicy źrenicy i skok pozycji Y to **artefakty**, a nie prawdziwe zachowanie oka.

In [ ]:
mrugniecia = luki[tabela_luk["rodzaj"].to_numpy() == "mrugnięcie"]
poczatek, koniec = mrugniecia[0]
okno = slice(max(poczatek - 150, 0), koniec + 150)  # ±300 ms wokół mrugnięcia

fig, osie = plt.subplots(2, 1, figsize=(10, 5), sharex=True)
osie[0].plot(surowe["t_s"][okno], surowe["y_px"][okno], ".-", ms=3)
osie[0].set_ylabel("Y [px]")
osie[1].plot(surowe["t_s"][okno], surowe["zrenica_mm"][okno], ".-", ms=3, color=MORSKI)
osie[1].set_ylabel("Źrenica [mm]")
for os_ in osie:
    os_.axvspan(surowe["t_s"][poczatek], surowe["t_s"][koniec - 1], color="red", alpha=0.2, label="luka w danych")
osie[0].legend()
osie[1].set_xlabel("Czas [s]")
fig.suptitle("Zbliżenie na mrugnięcie – artefakty na brzegach luki")
plt.tight_layout()
plt.show()

Poszerzamy każde mrugnięcie o margines (✏️ możesz zmienić te wartości), oznaczamy te próbki jako brakujące, a następnie **interpolujemy liniowo** wszystkie luki. Zapamiętujemy przy tym maskę `brak_danych` – algorytm detekcji fiksacji nie powinien „wierzyć” danym odtworzonym przez interpolację.

In [ ]:
MARGINES_PRZED_MS = 50  # ✏️
MARGINES_PO_MS = 100  # ✏️

dane = surowe.copy()
maska_mrugniec = np.zeros(len(dane), dtype=bool)
for poczatek, koniec in mrugniecia:
    p = max(poczatek - MARGINES_PRZED_MS * FS // 1000, 0)
    k = min(koniec + MARGINES_PO_MS * FS // 1000, len(dane))
    maska_mrugniec[p:k] = True

dane.loc[maska_mrugniec, ["x_px", "y_px", "zrenica_mm"]] = np.nan
dane["mrugniecie"] = maska_mrugniec
dane["brak_danych"] = dane["x_px"].isna()  # mrugnięcia (z marginesami) + krótkie utraty sygnału

for kolumna in ["x_px", "y_px", "zrenica_mm"]:
    dane[kolumna] = dane[kolumna].interpolate(method="linear", limit_direction="both")

print(f"Próbki oznaczone jako mrugnięcia: {dane['mrugniecie'].sum()} ({100 * dane['mrugniecie'].mean():.1f}%)")
print(f"Wszystkie próbki bez wiarygodnych danych: {dane['brak_danych'].sum()}")
print("Pozostałe braki po interpolacji:", int(dane[["x_px", "y_px", "zrenica_mm"]].isna().sum().sum()))

In [ ]:
fig, osie = plt.subplots(2, 1, figsize=(12, 6), sharex=True)
osie[0].plot(surowe["t_s"], surowe["zrenica_mm"], color="lightgray", lw=2, label="surowe")
osie[0].plot(dane["t_s"], dane["zrenica_mm"], color=MORSKI, lw=1, label="po usunięciu mrugnięć")
osie[0].set_ylabel("Źrenica [mm]")
osie[0].legend()
osie[1].plot(surowe["t_s"], surowe["y_px"], color="lightgray", lw=2, label="surowe")
osie[1].plot(dane["t_s"], dane["y_px"], color=NIEBIESKI, lw=1, label="po usunięciu mrugnięć")
osie[1].set_ylabel("Y [px]")
osie[1].set_xlabel("Czas [s]")
osie[1].legend()
fig.suptitle("Efekt usunięcia mrugnięć i interpolacji")
plt.tight_layout()
plt.show()

## 4. Wygładzanie sygnału

Każdy eye-tracker ma **szum pomiarowy** – nawet gdy oko jest nieruchome, kolejne próbki różnią się o ułamki stopnia. Przy liczeniu prędkości szum ten jest silnie wzmacniany (różniczkowanie wzmacnia wysokie częstotliwości), dlatego przed detekcją sakad sygnał wygładzamy.

Dla pozycji wzroku często stosuje się **filtr Savitzky'ego–Golaya**: w przesuwającym się oknie dopasowuje on wielomian niskiego stopnia. Dobrze usuwa szum, a przy tym zachowuje kształt szybkich zmian, takich jak sakady (Nyström i Holmqvist, 2010). Średnicę źrenicy, która zmienia się powoli, wygładzamy filtrem dolnoprzepustowym (4 Hz).

In [ ]:
OKNO_SG_MS = 20  # ✏️ długość okna filtru Savitzky'ego–Golaya [ms]
okno_sg = int(OKNO_SG_MS * FS / 1000) | 1  # liczba próbek musi być nieparzysta

dane["x_wygl"] = savgol_filter(dane["x_px"], window_length=okno_sg, polyorder=2)
dane["y_wygl"] = savgol_filter(dane["y_px"], window_length=okno_sg, polyorder=2)

b, a = butter(N=3, Wn=4, fs=FS, btype="low")  # filtr Butterwortha, częstotliwość odcięcia 4 Hz
dane["zrenica_wygl"] = filtfilt(b, a, dane["zrenica_mm"])  # filtfilt = filtracja bez przesunięcia fazowego

print(f"Okno filtru Savitzky'ego–Golaya: {okno_sg} próbek ({okno_sg * 1000 / FS:.0f} ms)")

In [ ]:
# Zbliżenia: sakada (lewy panel) i fragment fiksacji (prawy panel, mocno powiększona oś Y)
fig, osie = plt.subplots(1, 2, figsize=(12, 4))
for os_, (t0, t1), tytul in zip(osie, [(1.4, 1.75), (1.55, 1.75)], ["Sakada", "Fiksacja (powiększenie)"]):
    fragment = (dane["t_s"] > t0) & (dane["t_s"] < t1)
    os_.plot(dane.loc[fragment, "t_s"], dane.loc[fragment, "x_px"], ".", ms=4, color="gray",
             label="po interpolacji (z szumem)")
    os_.plot(dane.loc[fragment, "t_s"], dane.loc[fragment, "x_wygl"], color=NIEBIESKI, lw=2, label="po wygładzeniu")
    os_.set(xlabel="Czas [s]", ylabel="X [px]", title=tytul)
osie[0].legend()
fig.suptitle("Filtr Savitzky'ego–Golaya – zbliżenie")
plt.tight_layout()
plt.show()

### Średnica źrenicy po oczyszczeniu

Po usunięciu mrugnięć i wygładzeniu widać wyraźnie **odruch źreniczny na światło**: plakat jest jaśniejszy niż szary ekran wyświetlany przed nim, więc źrenica zwęża się z opóźnieniem ok. 0,3 s, a największe zwężenie następuje po ok. 1 s. W badaniach pupilometrycznych zmiany średnicy zwykle odnosimy do **linii bazowej** (np. średniej z pierwszych 200 ms).

In [ ]:
linia_bazowa = dane.loc[dane["t_s"] < 0.2, "zrenica_wygl"].mean()
dane["zrenica_wzgledna"] = dane["zrenica_wygl"] - linia_bazowa

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(dane["t_s"], dane["zrenica_wzgledna"], color=MORSKI)
ax.axhline(0, color="black", lw=0.8)
ax.set(xlabel="Czas od pojawienia się bodźca [s]", ylabel="Zmiana średnicy [mm]",
       title=f"Zmiana średnicy źrenicy względem linii bazowej ({linia_bazowa:.2f} mm)")
plt.show()

## 5. Prędkość kątowa ruchu oka

Algorytm I-VT klasyfikuje każdą próbkę na podstawie **prędkości kątowej** [°/s]. Przeliczamy pozycję na stopnie kąta widzenia, a następnie liczymy pochodną (różnice centralne, `np.gradient`) i długość wektora prędkości:

$$v = \sqrt{\left(\frac{dx}{dt}\right)^2 + \left(\frac{dy}{dt}\right)^2}$$

Porównamy prędkość policzoną z sygnału **niewygładzonego** i **wygładzonego**.

In [ ]:
dane["x_deg"], dane["y_deg"] = px_na_stopnie(dane["x_wygl"], dane["y_wygl"])
dane["predkosc"] = np.hypot(np.gradient(dane["x_deg"]), np.gradient(dane["y_deg"])) * FS

# Dla porównania: prędkość z danych niewygładzonych (tylko po interpolacji)
x_surowe_deg, y_surowe_deg = px_na_stopnie(dane["x_px"], dane["y_px"])
predkosc_bez_wygladzania = np.hypot(np.gradient(x_surowe_deg), np.gradient(y_surowe_deg)) * FS

PROG_PREDKOSCI = 30  # ✏️ próg I-VT [°/s] – typowa wartość (np. filtr I-VT firmy Tobii)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(dane["t_s"], predkosc_bez_wygladzania, color="lightgray", lw=0.8, label="bez wygładzania")
ax.plot(dane["t_s"], dane["predkosc"], color=NIEBIESKI, lw=1, label="po wygładzeniu")
ax.axhline(PROG_PREDKOSCI, color="#444444", ls="--", label=f"próg {PROG_PREDKOSCI}°/s")
ax.set(xlim=(1, 4), ylim=(0, 150), xlabel="Czas [s]", ylabel="Prędkość [°/s]",
       title="Prędkość kątowa – zbliżenie (oś Y przycięta do 150°/s)")
ax.legend(loc="upper right")
plt.show()

for nazwa, v in [("bez wygładzania", predkosc_bez_wygladzania), ("po wygładzeniu", dane["predkosc"])]:
    print(f"Mediana prędkości ({nazwa}): {np.median(v):5.1f}°/s, "
          f"próbki powyżej progu: {100 * np.mean(v > PROG_PREDKOSCI):4.1f}%")

Bez wygładzania szum co chwilę przekracza próg 30°/s – algorytm „widziałby” sakady w środku fiksacji. Po wygładzeniu prędkość podczas fiksacji spada do kilku °/s, a próg przekraczają niemal wyłącznie prawdziwe sakady.

## 6. Detekcja fiksacji i sakad algorytmem I-VT

Algorytm **I-VT** (Salvucci i Goldberg, 2000) to jeden z najprostszych i najczęściej stosowanych sposobów detekcji fiksacji:

1. próbki o prędkości **poniżej progu** to kandydaci na fiksacje, **powyżej progu** – sakady,
2. kolejne próbki „fiksacyjne” łączymy w fiksacje,
3. sąsiednie fiksacje oddzielone bardzo krótką przerwą (< 75 ms) i leżące blisko siebie (< 0,5°) scalamy – przerwa to zwykle chwilowy wzrost szumu,
4. odrzucamy zbyt krótkie fiksacje (< 60 ms).

Kroki 3–4 pochodzą z opisu filtru I-VT firmy Tobii (Olsen, 2012). Próbek odtworzonych przez interpolację (`brak_danych`) nie zaliczamy ani do fiksacji, ani do sakad, a fiksacji rozdzielonych mrugnięciem nie scalamy.

In [ ]:
MAKS_PRZERWA_MS = 75  # łączymy fiksacje rozdzielone krótszą przerwą...
MAKS_ODLEGLOSC_DEG = 0.5  # ...jeśli ich środki leżą bliżej niż 0,5°
MIN_CZAS_FIKSACJI_MS = 60  # krótsze fiksacje odrzucamy


def wykryj_fiksacje_ivt(dane, prog=PROG_PREDKOSCI, maks_przerwa_ms=MAKS_PRZERWA_MS,
                        maks_odl_deg=MAKS_ODLEGLOSC_DEG, min_czas_ms=MIN_CZAS_FIKSACJI_MS,
                        kolumna_predkosci="predkosc"):
    """Wykrywa fiksacje algorytmem I-VT i zwraca je w postaci tabeli."""
    t = dane["t_s"].to_numpy()
    x_deg, y_deg = dane["x_deg"].to_numpy(), dane["y_deg"].to_numpy()
    brak = dane["brak_danych"].to_numpy()
    mrugniecie = dane["mrugniecie"].to_numpy()

    # Krok 1–2: ciągłe odcinki próbek o prędkości poniżej progu
    odcinki = znajdz_odcinki((dane[kolumna_predkosci].to_numpy() < prog) & ~brak)
    if len(odcinki) == 0:
        return pd.DataFrame()

    # Krok 3: scalanie sąsiednich, bliskich sobie fiksacji (ale nigdy ponad mrugnięciem)
    scalone = [list(odcinki[0])]
    for p, k in odcinki[1:]:
        pp, pk = scalone[-1]
        przerwa_ms = (t[p] - t[pk - 1]) * 1000
        odleglosc = np.hypot(x_deg[p:k].mean() - x_deg[pp:pk].mean(), y_deg[p:k].mean() - y_deg[pp:pk].mean())
        if przerwa_ms < maks_przerwa_ms and odleglosc < maks_odl_deg and not mrugniecie[pk:p].any():
            scalone[-1][1] = k
        else:
            scalone.append([p, k])

    # Krok 4: tabela fiksacji bez zbyt krótkich
    wiersze = []
    for p, k in scalone:
        czas_ms = (k - p) * 1000 / FS
        if czas_ms >= min_czas_ms:
            wiersze.append({
                "start_s": t[p], "koniec_s": t[k - 1] + 1 / FS, "czas_trwania_ms": czas_ms,
                "x_px": dane["x_wygl"].iloc[p:k].mean(), "y_px": dane["y_wygl"].iloc[p:k].mean(),
            })
    return pd.DataFrame(wiersze)


def wykryj_sakady(dane, prog=PROG_PREDKOSCI, min_czas_ms=10):
    """Zwraca tabelę sakad: ciągłe odcinki próbek o prędkości powyżej progu."""
    t, v = dane["t_s"].to_numpy(), dane["predkosc"].to_numpy()
    x_deg, y_deg = dane["x_deg"].to_numpy(), dane["y_deg"].to_numpy()
    wiersze = []
    for p, k in znajdz_odcinki((v >= prog) & ~dane["brak_danych"].to_numpy()):
        czas_ms = (k - p) * 1000 / FS
        if czas_ms < min_czas_ms:
            continue
        p0, k0 = max(p - 1, 0), min(k, len(t) - 1)  # pozycja tuż przed i tuż po przekroczeniu progu
        wiersze.append({
            "start_s": t[p], "czas_trwania_ms": czas_ms,
            "amplituda_deg": np.hypot(x_deg[k0] - x_deg[p0], y_deg[k0] - y_deg[p0]),
            "predkosc_szczytowa": v[p:k].max(),
        })
    return pd.DataFrame(wiersze)


fiksacje = wykryj_fiksacje_ivt(dane)
sakady = wykryj_sakady(dane)
print(f"Wykryto {len(fiksacje)} fiksacji i {len(sakady)} sakad")
fiksacje.head(10).round(2)

In [ ]:
print("Fiksacje – statystyki opisowe:")
print(fiksacje["czas_trwania_ms"].describe().round(1).to_string())
print("\nSakady – statystyki opisowe:")
print(sakady[["czas_trwania_ms", "amplituda_deg", "predkosc_szczytowa"]].describe().round(1).to_string())

### Wizualizacja wyników detekcji

Na wykresie pozycji w czasie zaznaczamy wykryte fiksacje (zielone pasy) i mrugnięcia (szare pasy). Dobra detekcja powinna pokrywać każdy płaski fragment sygnału jedną fiksacją.

In [ ]:
fig, osie = plt.subplots(3, 1, figsize=(12, 8), sharex=True)
osie[0].plot(dane["t_s"], dane["x_wygl"], color=NIEBIESKI, label="X")
osie[0].plot(dane["t_s"], dane["y_wygl"], color=POMARANCZOWY, label="Y")
osie[0].set_ylabel("Pozycja [px]")
osie[0].legend(loc="upper right")
osie[1].plot(dane["t_s"], dane["predkosc"], color="black", lw=0.8)
osie[1].axhline(PROG_PREDKOSCI, color="#444444", ls="--")
osie[1].set(ylabel="Prędkość [°/s]", yscale="log", ylim=(1, 1000))
osie[2].plot(dane["t_s"], dane["zrenica_wygl"], color=MORSKI)
osie[2].set(ylabel="Źrenica [mm]", xlabel="Czas [s]")
for os_ in osie:
    for _, f in fiksacje.iterrows():
        os_.axvspan(f["start_s"], f["koniec_s"], color=MORSKI, alpha=0.2, lw=0)
    for p, k in znajdz_odcinki(dane["mrugniecie"]):
        os_.axvspan(dane["t_s"].iloc[p], dane["t_s"].iloc[k - 1], color="gray", alpha=0.4, lw=0)
fig.suptitle("Wynik algorytmu I-VT: fiksacje (zielone) i mrugnięcia (szare)")
plt.tight_layout()
plt.show()

A tak wyglądają wykryte fiksacje na tle bodźca. Wielkość koła odpowiada czasowi trwania fiksacji – to zapowiedź wizualizacji, którymi zajmiemy się w notebooku 02.

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6.75))
ax.imshow(obraz, extent=[0, EKRAN_SZER_PX, EKRAN_WYS_PX, 0])
ax.plot(fiksacje["x_px"], fiksacje["y_px"], color="white", lw=1, alpha=0.8)
ax.scatter(fiksacje["x_px"], fiksacje["y_px"], s=fiksacje["czas_trwania_ms"], color=NIEBIESKI,
           alpha=0.7, edgecolor="white")
ax.set(xlim=(0, EKRAN_SZER_PX), ylim=(EKRAN_WYS_PX, 0), title="Fiksacje wykryte algorytmem I-VT")
ax.grid(False)
plt.show()

## 7. Walidacja wyników

### 7.1. Główna sekwencja sakad

Sakady mają stałą „sygnaturę”: im większa amplituda, tym większa prędkość szczytowa (zależność potęgowa, która nasyca się przy ok. 500–700°/s). To tzw. **główna sekwencja** (*main sequence*; Bahill i in., 1975). Jeśli wykryte „sakady” leżą daleko od tej zależności, prawdopodobnie są artefaktami (np. pozostałościami po mrugnięciach).

In [ ]:
wsp_b, wsp_a = np.polyfit(np.log10(sakady["amplituda_deg"]), np.log10(sakady["predkosc_szczytowa"]), 1)
amplitudy = np.logspace(np.log10(sakady["amplituda_deg"].min()), np.log10(sakady["amplituda_deg"].max()), 50)

fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(sakady["amplituda_deg"], sakady["predkosc_szczytowa"], alpha=0.7)
ax.plot(amplitudy, 10**wsp_a * amplitudy**wsp_b, color=POMARANCZOWY,
        label=f"dopasowanie: $v = {10**wsp_a:.0f} \\cdot A^{{{wsp_b:.2f}}}$")
ax.set(xscale="log", yscale="log", xlabel="Amplituda sakady [°]", ylabel="Prędkość szczytowa [°/s]",
       title="Główna sekwencja sakad")
ax.legend()
plt.show()

### 7.2. Jak wynik zależy od progu prędkości?

Próg I-VT jest parametrem wybieranym przez badacza, więc warto sprawdzić, jak bardzo wyniki od niego zależą. Powtarzamy detekcję dla różnych progów w trzech wariantach: pełny algorytm (wygładzanie + scalanie fiksacji) oraz dwa warianty uproszczone, bez kroku scalania – na danych wygładzonych i niewygładzonych.

In [ ]:
dane["predkosc_bez_wygladzania"] = predkosc_bez_wygladzania
warianty = {
    "wygładzanie + scalanie (pełny I-VT)": dict(kolumna_predkosci="predkosc"),
    "wygładzanie, bez scalania": dict(kolumna_predkosci="predkosc", maks_przerwa_ms=0),
    "bez wygładzania i bez scalania": dict(kolumna_predkosci="predkosc_bez_wygladzania", maks_przerwa_ms=0),
}
wyniki_progow = []
for prog in [10, 15, 20, 30, 40, 60, 80, 100, 150]:
    for opis, parametry in warianty.items():
        f = wykryj_fiksacje_ivt(dane, prog=prog, **parametry)
        wyniki_progow.append({"próg [°/s]": prog, "wariant": opis, "liczba fiksacji": len(f),
                              "średni czas fiksacji [ms]": f["czas_trwania_ms"].mean() if len(f) else np.nan})
wyniki_progow = pd.DataFrame(wyniki_progow)

fig, osie = plt.subplots(1, 2, figsize=(13, 4.5))
for opis in warianty:
    grupa = wyniki_progow[wyniki_progow["wariant"] == opis]
    osie[0].plot(grupa["próg [°/s]"], grupa["liczba fiksacji"], "o-", label=opis)
    osie[1].plot(grupa["próg [°/s]"], grupa["średni czas fiksacji [ms]"], "o-", label=opis)
osie[0].set(xlabel="Próg prędkości [°/s]", ylabel="Liczba fiksacji")
osie[1].set(xlabel="Próg prędkości [°/s]", ylabel="Średni czas fiksacji [ms]")
for os_ in osie:
    os_.axvline(PROG_PREDKOSCI, color="#444444", ls="--", lw=1)
osie[0].legend()
fig.suptitle("Wrażliwość algorytmu I-VT na wybór progu")
plt.tight_layout()
plt.show()

wyniki_progow.pivot(index="próg [°/s]", columns="wariant", values="liczba fiksacji")

Pełny algorytm daje niemal ten sam wynik w szerokim zakresie progów – to dobra wiadomość. Bez kroku scalania wynik bardziej zależy od progu, a bez wygładzania algorytm wręcz się „sypie”: przy niskich progach szum dzieli fiksacje na kawałki krótsze niż 60 ms, które są odrzucane (w skrajnym przypadku nie zostaje ani jedna fiksacja!). **Wygładzanie i scalanie to dwa niezależne zabezpieczenia przed szumem.**

### 7.3. Porównanie z „prawdą”

Ponieważ dane są symulowane, znamy prawdziwe fiksacje tej osoby. Zapisano je w pliku z raportem fiksacji całej grupy (`et_fiksacje_grupa.csv`, uczestnik `U01`), z którego skorzystamy w notebooku 02. Porównajmy je z wynikiem naszego algorytmu.

In [ ]:
prawda = pd.read_csv(DANE / "et_fiksacje_grupa.csv").query("uczestnik == 'U01'")
print(f"Prawdziwe fiksacje: {len(prawda)}, średni czas trwania {prawda['czas_trwania_ms'].mean():.0f} ms")
print(f"Wykryte fiksacje:   {len(fiksacje)}, średni czas trwania {fiksacje['czas_trwania_ms'].mean():.0f} ms")

fig, ax = plt.subplots(figsize=(12, 2.8))
ax.broken_barh(list(zip(prawda["start_ms"] / 1000, prawda["czas_trwania_ms"] / 1000)), (1.1, 0.8),
               color="gray", edgecolor="white")
ax.broken_barh(list(zip(fiksacje["start_s"], fiksacje["czas_trwania_ms"] / 1000)), (0.1, 0.8),
               color=MORSKI, edgecolor="white")
for p, k in znajdz_odcinki(dane["mrugniecie"]):
    ax.axvspan(dane["t_s"].iloc[p], dane["t_s"].iloc[k - 1], color="red", alpha=0.25, lw=0)
ax.set(yticks=[0.5, 1.5], yticklabels=["I-VT", "prawda"], xlabel="Czas [s]",
       title="Fiksacje prawdziwe i wykryte (czerwone pasy – mrugnięcia)")
ax.grid(False)
plt.show()

Większość fiksacji odtworzyliśmy bardzo dokładnie. Różnice pojawiają się w dwóch sytuacjach:

* przy **mrugnięciach** – mrugnięcie w środku fiksacji dzieli ją na dwie części albo „zjada” krótką fiksację,
* przy **bardzo małych sakadach** – ich prędkość może nie przekroczyć progu, więc dwie sąsiednie fiksacje zlewają się w jedną.

To normalne ograniczenia algorytmu I-VT. W raportach z badań zawsze podajemy parametry detekcji i sposób traktowania mrugnięć.

## 8. Zapis wyników

Zapisujemy tabelę fiksacji i oczyszczone próbki do katalogu `wyniki/`.

In [ ]:
fiksacje.round(3).to_csv(WYNIKI / "01_fiksacje_U01.csv", index=False)
sakady.round(3).to_csv(WYNIKI / "01_sakady_U01.csv", index=False)
dane[["t_s", "x_wygl", "y_wygl", "zrenica_wygl", "predkosc", "mrugniecie", "brak_danych"]].round(4).to_csv(
    WYNIKI / "01_probki_oczyszczone_U01.csv", index=False)
print("Zapisano pliki:", *sorted(p.name for p in WYNIKI.glob("01_*")), sep="\n  ")

## Podsumowanie

* Surowe dane eye-trackingowe wymagają kontroli jakości: sprawdzamy częstotliwość próbkowania i odsetek utraconych danych.
* Mrugnięcia usuwamy **razem z marginesem**, bo brzegi luk zawierają artefakty (fałszywy spadek źrenicy, skoki pozycji).
* Przed liczeniem prędkości sygnał trzeba wygładzić – inaczej szum udaje sakady.
* Algorytm I-VT jest prosty i skuteczny, ale jego wyniki zależą od parametrów (próg, scalanie, minimalny czas fiksacji). Zawsze je raportujemy.
* Główna sekwencja sakad to szybki test poprawności detekcji.

## Zadania do samodzielnego wykonania

**Zadanie 1.** Zmień próg `PROG_PREDKOSCI` na 15°/s oraz 100°/s i uruchom ponownie sekcję 6. Jak zmienia się liczba i średni czas fiksacji? Które fiksacje „znikają” lub „dzielą się” jako pierwsze?

**Zadanie 2.** Ustaw `MARGINES_PRZED_MS = 0` i `MARGINES_PO_MS = 0`. Czy w danych pojawiły się nowe, podejrzane sakady? Sprawdź to na wykresie głównej sekwencji.

**Zadanie 3.** Zaimplementuj algorytm **I-DT** (*Dispersion-Threshold Identification*; Salvucci i Goldberg, 2000) i porównaj wyniki z I-VT.

*Wskazówka:* zacznij od okna o długości 100 ms; jeśli dyspersja $D = (\max x - \min x) + (\max y - \min y)$ jest mniejsza niż 1°, poszerzaj okno, dopóki $D$ nie przekroczy progu – całe okno to jedna fiksacja. W przeciwnym razie przesuń okno o jedną próbkę.

**Zadanie 4.** Na podstawie kolumny `zrenica_wzgledna` wyznacz moment i wielkość maksymalnego zwężenia źrenicy. Czy wynik zgadza się z typowym przebiegiem odruchu źrenicznego?

In [ ]:
# Twoje rozwiązania:

## Literatura

* Bahill, A. T., Clark, M. R., i Stark, L. (1975). The main sequence, a tool for studying human eye movements. *Mathematical Biosciences, 24*(3–4), 191–204.
* Holmqvist, K., Nyström, M., Andersson, R., Dewhurst, R., Jarodzka, H., i van de Weijer, J. (2011). *Eye tracking: A comprehensive guide to methods and measures*. Oxford University Press.
* Mathôt, S., Fabius, J., Van Heusden, E., i Van der Stigchel, S. (2018). Safe and sensible preprocessing and baseline correction of pupil-size data. *Behavior Research Methods, 50*(1), 94–106.
* Nyström, M., i Holmqvist, K. (2010). An adaptive algorithm for fixation, saccade, and glissade detection in eyetracking data. *Behavior Research Methods, 42*(1), 188–204.
* Olsen, A. (2012). *The Tobii I-VT fixation filter: Algorithm description*. Tobii Technology.
* Salvucci, D. D., i Goldberg, J. H. (2000). Identifying fixations and saccades in eye-tracking protocols. *Proceedings of the 2000 Symposium on Eye Tracking Research & Applications*, 71–78.